In [11]:
%load_ext autoreload
%autoreload 2

# Run this cell first to prevent cached modules!

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# RAG Ingestion Pipeline Verification

This notebook verifies the `RAGIngestion` class as required by the Stage 1 Prototyping rules.

In [12]:
import sys
import os
sys.path.append("../../..")
from google.cloud import bigquery
from pipelines.enterprise_knowledge_base.rag_ingestion import RAGIngestion
from pipelines.enterprise_knowledge_base.orchestrator import KBIngestionPipeline

In [13]:
# Initialize the pipeline orchestrator
PROJECT_ID = "ag-core-dev-fdx7"
pipeline = KBIngestionPipeline(project_id=PROJECT_ID)


In [14]:
# Run the full pipeline (staging + vectorization)
GCS_URI = "gs://kb-landing-zone-test/ingested/Gumtree - SOW WIP.pdf"

try:
    pipeline.trigger_pipeline(GCS_URI)
except Exception as e:
    print(f'Pipeline execution result: {e}')

Triggering pipeline for gs://kb-landing-zone-test/ingested/Gumtree - SOW WIP.pdf...
Skipping Document Classification Phase for testing...
Starting RAG Ingestion for gs://kb-landing-zone-test/ingested/Gumtree - SOW WIP.pdf...
Successfully staged 40 chunks.
Initiating BigQuery ML vectorization...
Vectorization complete.


In [15]:
# Verify BigQuery: Check if embeddings were generated
bq_client = bigquery.Client(project=PROJECT_ID)
query = f"""
SELECT chunk_id, gcs_uri, ARRAY_LENGTH(embedding) as embedding_length 
FROM `{PROJECT_ID}.knowledge_base.documents_chunks` 
WHERE gcs_uri = 'gs://kb-landing-zone-test/ingested/Gumtree - SOW WIP.pdf'
LIMIT 5
"""

results = bq_client.query(query).result()
for row in results:
     print(dict(row))

{'chunk_id': '735a7762-ac2c-4464-a55a-066d28d26f1a', 'gcs_uri': 'gs://kb-landing-zone-test/ingested/Gumtree - SOW WIP.pdf', 'embedding_length': 1408}
{'chunk_id': 'c296cfbb-9e1a-4b56-ad13-ba6426fb4ce9', 'gcs_uri': 'gs://kb-landing-zone-test/ingested/Gumtree - SOW WIP.pdf', 'embedding_length': 1408}
{'chunk_id': '43886d2f-a712-4074-8feb-1d81964e56c6', 'gcs_uri': 'gs://kb-landing-zone-test/ingested/Gumtree - SOW WIP.pdf', 'embedding_length': 1408}
{'chunk_id': '5eb73606-ba17-40a8-a675-afc359c29bfe', 'gcs_uri': 'gs://kb-landing-zone-test/ingested/Gumtree - SOW WIP.pdf', 'embedding_length': 0}
{'chunk_id': '2d9bf0f9-75e5-4cf8-b542-be6e870026a7', 'gcs_uri': 'gs://kb-landing-zone-test/ingested/Gumtree - SOW WIP.pdf', 'embedding_length': 1408}


## Cloud Function Endpoint Verification
Test the deployed Cloud Function endpoint by sending an HTTP POST request.

In [20]:
import requests
import google.auth
from google.auth.transport.requests import Request as GoogleAuthRequest

# Fetch identity token for authentication
credentials, project = google.auth.default()
auth_request = GoogleAuthRequest()
credentials.refresh(auth_request)
token = credentials.token

# Define Cloud Function URL (Replace with actual deployment URL)
CLOUD_FUNCTION_URL = "https://rag-ingestion-function-o2u6kmb3eq-uc.a.run.app"

headers = {
    "Authorization": f"Bearer {token}",
    "Content-Type": "application/json"
}

payload = {
    "gcs_uri": GCS_URI
}

print(f"Sending request to {CLOUD_FUNCTION_URL}...")
response = requests.post(CLOUD_FUNCTION_URL, json=payload, headers=headers)

print(f"Status Code: {response.status_code}")
print(f"Response: {response.text}")


Sending request to https://rag-ingestion-function-o2u6kmb3eq-uc.a.run.app...
Status Code: 500
Response: {"error":"Document gs://kb-landing-zone-test/ingested/Gumtree - SOW WIP.pdf has already been processed."}

